In [1]:
from langchain_community.document_loaders import UnstructuredURLLoader

C:\Users\ADITHYA UBALE\AppData\Local\Temp\ipykernel_23132\2420766737.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredURLLoader
c:\Users\ADITHYA UBALE\miniconda3\envs\test\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
urls = [
    'https://www.victoriaonmove.com.au/local-removalists.html',
    'https://victoriaonmove.com.au/index.html',
    'https://victoriaonmove.com.au/contact.html'
]

loaders = UnstructuredURLLoader(urls=urls)
data = loaders.load()

In [14]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n"],
    chunk_size=1000,
    chunk_overlap=0
)
docs = text_splitter.split_documents(data)

print(f"Total Length of the docs : {len(docs)}")

Total Length of the docs : 12


In [15]:
docs[0]

Document(metadata={'source': 'https://www.victoriaonmove.com.au/local-removalists.html'}, page_content='Please wait while your request is being verified...')

In [17]:
docs[1]

Document(metadata={'source': 'https://victoriaonmove.com.au/index.html'}, page_content='★★★★★ 5.0 · 111 Google Reviews\n\nRelocate with Confidence\n\nYour trusted partner in seamless moving and packing solutions — serving Melbourne and all of Australia.\n\nGet a Free Quote Call 0404 922 328\n\n5.0★\n\nGoogle Rating\n\n111\n\nVerified Reviews\n\nInterstate Routes\n\n$125\n\nStarting /hr\n\nVictoria On Move team carefully wrapping furniture for a move\n\nMoving boxes neatly stacked inside a Victoria On Move truck\n\nVictoria On Move removalists carefully moving large furniture items\n\nOur Fleet\n\nTransparent, Affordable Pricing\n\nAll trucks come with 2 experienced movers, trolleys, blankets, and loading ramps. No hidden fees.\n\nVictoria On Move small truck — 4.5 ton, ideal for 1-bedroom moves\n\nMost Popular\n\nSmall Truck\n\n$125/hr\n\nWith 2 Movers\n\n4.5 ton · 20 m³\n\nIdeal for student apartments, small offices, and 1-bedroom homes.\n\nBook This Truck →\n\nVictoria On Move medium

In [19]:
!pip install langchain-huggingface sentence-transformers

  Using cached langchain_huggingface-1.2.2-py3-none-any.whl.metadata (4.0 kB)
Using cached langchain_huggingface-1.2.2-py3-none-any.whl (31 kB)


In [22]:
from langchain_chroma import Chroma
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_huggingface import HuggingFaceEmbeddings


## Create an embeddings using HF
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4493.40it/s]


In [23]:
from dotenv import load_dotenv

load_dotenv()

True

In [26]:
#vector stores which store the data into vectors
vector_stores = Chroma.from_documents(documents=docs, embedding=embeddings)

#Retreiver to reterive the similar data
reteriver = vector_stores.as_retriever(search_type = "similarity", search_kwargs={"k":3})

In [27]:
reterived_docs = reteriver.invoke("What Kind of services they provide")

In [30]:
print(len(reterived_docs))

3


In [31]:
print(reterived_docs[0].page_content)

Long-haul Cairns interstate removals handled with care, full insurance, and reliable delivery.

Book Now →

Real Moves. Real Team.

Watch Us in Action

No actors, no stock footage — just our crew doing what we do best, every single day.

Moving Day

Wrapping & Packing

Loading Up

On the Road

Delivery

Happy Clients

Furniture Care

Team at Work

Interstate Run

Hover or tap any video to play · All footage is real — our team, our trucks, our work

Our Commitment

Our Moving Work & Ethics

Moving house furniture is an art that comes with experience and a steadfast commitment to our customers' needs. At Victoria On Move, we approach every relocation with the same dedication — whether it's a single bedroom apartment or a large family home.

Unlike some other moving companies, we value the trust our customers place in us. Our customer reviews on Google are genuine, and we never manipulate them. Every 5-star review is earned through hard work and exceptional service.

Fully Insured


In [34]:
#Create an LLM USING NVIDIA OPENAI_Endpoint
import os

llm = ChatNVIDIA(
    model = "openai/gpt-oss-20b",
    api_key = os.environ['NVIDIA_API_KEY'],
    max_tokens=200,
    temperature=0.2,
    top_p=1
)

C:\Users\ADITHYA UBALE\AppData\Local\Temp\ipykernel_23132\3281422437.py:4: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  llm = ChatNVIDIA(


In [41]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt ),
        ("human", "{input}"),
    ]
)

In [42]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(reteriver, question_answer_chain)

In [43]:
response = rag_chain.invoke({"input": "What kind of service they provide"})
print(response["answer"])

Victoria On Move offers full‑service residential moving and removal solutions, handling everything from packing and loading to long‑haul interstate transport and final delivery. They provide fully insured, transparent‑priced moves with no hidden fees. Their crew works on every step of the relocation, ensuring furniture and belongings are protected and delivered safely.
